In [ ]:
# Differential expression by condition and cluster renaming on the integrated object.
import scanpy as sc
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad
import scvelo as scv
import palantir as pl
import scanpy.external as sce

In [ ]:
!pwd

In [ ]:
adata

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/old_inte_palantir.h5ad')
adata

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="condition", 
                        method="wilcoxon", layer = 'data')


In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/home/yanjun2/scanpy/8_figures/condition.csv', index=False)

In [ ]:
#adata = sc.read('/home/yanjun2/scanpy/7/inte_palantir.h5ad')

In [ ]:
adata.obs['leiden_res_0_8'] = adata.obs['leiden_res_0_8'].map(
    {
        'Mac7' : 'Mac1',
        'Mac3' : 'Mac2',
        'Mac5' : 'Mac3',
        'Mac6' : 'Mac4',
        'Mac0' : 'Mac6',
        'Mac4' : 'Mac6',
        'Mac2' : 'Mac5',
        'Mac1' : 'Mac7'
    }
)

In [ ]:
adata.obsm = {}

In [ ]:
cell = ['CTGCCATAGTATGAAC-1_3']
umap = pd.DataFrame(adata.obsm['X_umap'][:,0:2], index=adata.obs_names, columns=['x', 'y'])
plt.scatter(umap["x"], umap["y"], s=5, color="lightgrey")
plt.scatter(umap.loc[cell, "x"], umap.loc[cell, "y"], s=30)

In [ ]:
# PAGA and diffusion maps
def run_pr(ad, cluster, startcell):
    sc.tl.paga(ad, groups = cluster)
    sce.tl.palantir(ad, n_components = 25, knn = 50, impute_data = False, use_adjacency_matrix = True)
    start = startcell
    pr_res = sce.tl.palantir_results(ad, knn = 50, early_cell = start)
    ad.obs['palantir_pseudotime'] = pr_res.pseudotime
    ad.obs['palantir_entropy'] = pr_res.entropy
    run_pr.pr_res = pr_res.branch_probs

In [ ]:
run_pr(adata, cluster = 'leiden_res_0_8', startcell = cell)

In [ ]:
sc.pl.umap(adata, color='palantir_pseudotime', color_map='viridis', show=True)

In [ ]:
sc.pl.violin(adata, 
             keys='palantir_pseudotime', 
             groupby='leiden_res_0_8', 
             order=['Mac1', 'Mac2', 'Mac3', 'Mac4', 'Mac5', 'Mac6', 'Mac7']
             )

In [ ]:
sc.pl.embedding(adata, basis='X_palantir_diff_comp', color='leiden_res_0_8', 
                    title='Palantir Diffusion Components', size=20)

In [ ]:
# Save result
adata.write('/Users/yanjun/Desktop/scanpy/7_integrated/inte_palantir.h5ad')